To run this, press "*Runtime*" and press "*Run all*" on a Google Colab A100 instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

### This Notebook: Continued Pretraining (CPT) on Agentic AI Book

This notebook is configured for **continued pretraining** of Gemma 4 on the "Agentic AI frameworks, platforms, protocols, and tools on AWS" book.

**Key differences from instruction fine-tuning:**
- Uses raw text completion (next-token prediction), not chat format
- Trains on ALL tokens (no masking of user vs assistant parts)
- Includes `embed_tokens` and `lm_head` in LoRA for domain adaptation
- Uses `UNSLOTH_RETURN_LOGITS=1` to disable CCE (not supported for CPT)
- Higher LoRA rank (128) and rank-stabilized LoRA for better adaptation
- EOS tokens added to help the model learn proper stopping

**Data source:** 92 pages from the AWS Prescriptive Guidance PDF (OCR'd to markdown)

---

**Other Unsloth Resources:**

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"

# Set environment variable to disable CCE for continued pretraining
# CCE (Constant Cross Entropy) is not supported for CPT
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [2]:
from unsloth import FastModel
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
    device_map = "balanced", # Use 2x Tesla T4s on Kaggle
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

# Load the book text from individual page files
import os
import glob

pages_dir = "/Users/anton/Documents/FineTuning/ocr-playground-download-20260329T115537Z/agentic-ai-frameworks.pdf/pages"

# Get all page directories and sort them
page_dirs = sorted(glob.glob(os.path.join(pages_dir, "page-*")), 
                   key=lambda x: int(os.path.basename(x).replace("page-", "")))

print(f"Found {len(page_dirs)} pages")

# Load text from each page
all_pages = []
for page_dir in page_dirs:
    md_path = os.path.join(page_dir, "markdown.md")
    if os.path.exists(md_path):
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
            if content.strip():  # Only add non-empty pages
                all_pages.append(content)

print(f"Loaded {len(all_pages)} pages with content")

# Join all pages into one text (with page separators)
book_text = "\n\n".join(all_pages)
print(f"\nTotal book length: {len(book_text)} characters")
print(f"Approximate tokens: {len(book_text) // 4} tokens")

In [3]:
from datasets import load_dataset

dataset_output_path = '/Users/anton/Documents/FineTuning/books_combined_dataset.csv'
dataset = load_dataset('csv', data_files=dataset_output_path)['train']
print(f'Loaded dataset with {len(dataset)} examples')

Found 92 pages
Loaded 92 pages with content

Total book length: 162359 characters
Approximate tokens: 40589


In [26]:
from transformers import TextStreamer

# Helper function for chat inference (for testing before CPT)
def do_gemma_4_chat_inference(messages, max_new_tokens = 128):
    # Use apply_chat_template for chat messages
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")
    
    _ = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

# Helper function for continued pretraining inference (text completion)
def do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.7):
    """For CPT inference - direct text completion"""
    import torch
    from transformers import TextIteratorStreamer
    from threading import Thread
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # FIX: Use underlying tokenizer for text completion
    input_ids = tokenizer.tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=True).to(device)
    attention_mask = torch.ones_like(input_ids)
    
    streamer = TextIteratorStreamer(tokenizer.tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=temperature,
        top_p=0.9,
        top_k=50,
        do_sample=True,
        repetition_penalty=1.2,      # ADD THIS
        no_repeat_ngram_size=2,      # ADD THIS
    )
    
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    generated_text = ""
    for text in streamer:
        print(text, end="", flush=True)
        generated_text += text
    thread.join()
    print()
    return generated_text

Let's make a poem about sloths!

# Let's finetune Gemma 4!

You can finetune the vision and text parts for now through selection - the audio part can also be finetuned - we're working to make it selectable as well!

We now add LoRA adapters so we only need to update a small amount of parameters!

In [5]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # Turn off for text CPT
    finetune_language_layers   = True,   # Keep on for continued pretraining
    finetune_attention_modules = True,   # Good for CPT
    finetune_mlp_modules       = True,   # Keep on always
    
    # Additional modules for continued pretraining
    # embed_tokens and lm_head help the model adapt to new vocabulary/domain
    finetune_embedding_modules = True,   # Enable embedding fine-tuning for CPT
    finetune_lm_head          = True,    # Enable lm_head fine-tuning for CPT

    r = 128,           # Higher rank for CPT (128 recommended vs 8 for chat)
    lora_alpha = 32,   # Alpha = r/4 or r/2 is common for CPT
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = True,  # Rank-stabilized LoRA helps for CPT
)

### Data Prep for Continued Pretraining

For continued pretraining, we load raw text and train the model to predict the next token. We don't use chat templates - just raw text chunks.

Now we'll chunk the text into sequences for training. For continued pretraining, we create overlapping chunks to maximize data usage.

In [6]:
# The chunking has already been handled in create_dataset.ipynb.
# We directly use the loaded dataset.

max_seq_length = 8192  # Retained for tokenizer bounds check

print(f'Dataset size: {len(dataset)} examples')

Created 40 text chunks
Average chunk length: 8016 chars

Dataset size: 40 examples


Let's verify the data by looking at a sample chunk:

In [7]:
print(f"Sample chunk (first 500 chars):\n{'='*50}\n")
print(dataset[0]["text"][:500])
print(f"\n{'='*50}\n")
print(f"Sample chunk (middle of book, first 500 chars):\n{'='*50}\n")
print(dataset[len(dataset)//2]["text"][:500])

Sample chunk (first 500 chars):

aws

Agentic AI frameworks, platforms, protocols, and tools on AWS

# AWS Prescriptive Guidance

![img-0.jpeg](img-0.jpeg)

Copyright © 2026 Amazon Web Services, Inc. and/or its affiliates. All rights reserved.

AWS Prescriptive Guidance
Agentic AI frameworks, platforms, protocols, and tools on AWS

# AWS Prescriptive Guidance: Agentic AI frameworks, platforms, protocols, and tools on AWS

Copyright © 2026 Amazon Web Services, Inc. and/or its affiliates. All rights reserved.

Amazon's trademarks


Sample chunk (middle of book, first 500 chars):

oice of agent protocols represents a strategic decision that balances immediate development needs with long-term flexibility and interoperability. By prioritizing open protocols and creating appropriate abstraction layers, organizations can build agent systems that remain adaptable to evolving technologies while meeting current business requirements.

For most organizations, MCP represents a strong foundation du

For continued pretraining, we use the raw text directly without chat templates. The model will learn to predict the next token.

In [8]:
# For continued pretraining, we just use the text as-is
# No chat template formatting needed - this is pure next-token prediction

# Let's see the first example
dataset[0]["text"][:200]

'aws\n\nAgentic AI frameworks, platforms, protocols, and tools on AWS\n\n# AWS Prescriptive Guidance\n\n![img-0.jpeg](img-0.jpeg)\n\nCopyright © 2026 Amazon Web Services, Inc. and/or its affiliates. All rights'

The text is already in the correct format for continued pretraining - we just pass the raw text to the model and it learns to predict the next token.

In [9]:
EOS_TOKEN = tokenizer.eos_token

def add_eos_token(examples):
    """Add EOS token to each text chunk for proper next-token prediction"""
    texts = examples["text"]
    # Add EOS token to help model learn when to stop generating
    return { "text" : [text + EOS_TOKEN for text in texts] }

# Apply EOS token addition
dataset = dataset.map(add_eos_token, batched = True)

print(f"Dataset features: {dataset.features}")
print(f"Number of examples: {len(dataset)}")
print(f"EOS token: {repr(EOS_TOKEN)}")

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Dataset features: {'text': Value('string')}
Number of examples: 40
EOS token: '<eos>'


The dataset is ready. Each example contains a chunk of text that will be used for next-token prediction training.

In [10]:
# Look at an example
print(dataset[0]["text"][:300] + "...")

aws

Agentic AI frameworks, platforms, protocols, and tools on AWS

# AWS Prescriptive Guidance

![img-0.jpeg](img-0.jpeg)

Copyright © 2026 Amazon Web Services, Inc. and/or its affiliates. All rights reserved.

AWS Prescriptive Guidance
Agentic AI frameworks, platforms, protocols, and tools on AWS
...


<a name="Train"></a>
### Train the model for Continued Pretraining

For continued pretraining, we train on all tokens (no masking of user vs assistant parts). We train the model to predict the next token in the sequence.

In [11]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,        # Use ratio instead of steps for CPT
        num_train_epochs = 20,       # Multiple epochs over the book
        max_steps = -1,             # Use num_train_epochs instead
        learning_rate = 5e-5,       # Lower LR for CPT stability
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
        # Key for CPT: lower learning rate for embeddings
        # This is handled via Unsloth's special handling in the trainer
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/40 [00:00<?, ? examples/s]

For continued pretraining, we train on ALL tokens (not just responses). We skip the `train_on_responses_only` step since we want the model to learn the full text distribution.

In [12]:
# SKIP this for continued pretraining!
# We train on ALL tokens, not just responses

# trainer = train_on_responses_only(...)  # DO NOT USE for continued pretraining

print("Training on all tokens for continued pretraining...")
print("This will train the model to predict the next token in the book text.")

Training on all tokens for continued pretraining...
This will train the model to predict the next token in the book text.


Let's verify the training setup by checking a tokenized example:

In [13]:
# Tokenize and check the first example
sample_text = dataset[0]["text"]

# FIX: Use underlying tokenizer for CPT
input_ids = tokenizer.tokenizer.encode(sample_text, truncation=True, max_length=max_seq_length)
decoded = tokenizer.tokenizer.decode(input_ids)

print(f"Original length: {len(sample_text)} chars")
print(f"Tokenized length: {len(input_ids)} tokens")
print(f"\nFirst 200 chars of decoded:\n{decoded[:200]}...")

Original length: 8197 chars
Tokenized length: 1894 tokens

First 200 chars of decoded:
aws

Agentic AI frameworks, platforms, protocols, and tools on AWS

# AWS Prescriptive Guidance

![img-0.jpeg](img-0.jpeg)

Copyright © 2026 Amazon Web Services, Inc. and/or its affiliates. All rights...


Verify that labels match the input_ids (next-token prediction):

In [14]:
# Check that we're training on all tokens
# FIX: Access underlying tokenizer
input_ids = tokenizer.tokenizer.encode(dataset[0]["text"], truncation=True, max_length=max_seq_length)
print("Input IDs (first 10):", input_ids[:10])
print("Labels would be the same for next-token prediction")
print(f"\nFor continued pretraining, all {len(input_ids)} tokens will be used for training.")

Input IDs (first 10): [11457, 108, 24333, 525, 12498, 44362, 236764, 14175, 236764, 28463]
Labels would be the same for next-token prediction

For continued pretraining, all 1894 tokens will be used for training.


In [15]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
0.021 GB of memory reserved.


# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [16]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40 | Num Epochs = 20 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 202,702,848 of 5,325,880,864 (3.81% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,12.599028
20,6.501849
30,4.126436
40,3.145946
50,2.508508
60,1.962676
70,1.479624
80,1.051742
90,0.648199
100,0.363029


In [17]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1862.2516 seconds used for training.
31.04 minutes used for training.
Peak reserved memory = 11.145 GB.
Peak reserved memory for training = 11.124 GB.
Peak reserved memory % of max memory = 76.53 %.
Peak reserved memory for training % of max memory = 76.385 %.


<a name="Inference"></a>
### Inference for Continued Pretraining

For text completion inference after continued pretraining, we use the model as a text completion model (not chat). The model will continue text based on the patterns learned from the book.

In [25]:
# Prepare for inference
FastModel.for_inference(model)

# Test prompts related to the book's topic (Agentic AI on AWS)
test_prompts = [
    "Agentic AI frameworks on AWS include",
    "The Model Context Protocol (MCP) is",
    "When implementing multi-agent systems,",
    "Amazon Bedrock Agents provides",
    "What is Amazon Bedrock?",
    "what is multi-agent systems"
]

# Use the helper function for CPT text completion
prompt = test_prompts[5]
print(f"Prompt: {prompt}")
print("="*50)
print("Generated continuation:")
print("="*50)

do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.7)

Prompt: what is multi-agent systems
Generated continuation:
. Multi-agent systems are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent sys

'. Multi-agent systems are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent systems that are multi-agent syst

### Try more prompts

You can test the model with different prompts to see how well it learned the book's content:

In [27]:
# Test with different prompts
for prompt in test_prompts[0:]:
    print(f"\nPrompt: {prompt}")
    print("-" * 50)
    
    do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.7)


Prompt: Agentic AI frameworks on AWS include
--------------------------------------------------
 AWS Identity Object (IAM) Permissions.

AWS Identity and Access Management (Identity and access management) AWS IAM Identity-Object (Idp) Credentials allows you to use your identity provider (identity provider or Idp), such as Amazon Cognito, to log in to AWS services with your credentials. For more information, see Using an Identity Provider for AWS Services in the Identity Federation Guide.

Prompt: The Model Context Protocol (MCP) is
--------------------------------------------------
 a new MCPD-based MCP version.

This MCP specification updates the MCP 5.0 MCPR-1.2 MCPs and MCPRs specifications. It also replaces the MCR 3.4 MCPM specification. This MCP implementation supports MCP versions 6.x, 7.xx, and 8.xxx. The MCP support was added to MCP Version 9.1 and later. For more information about MCP Support for MCP Versions 4.X through 10.XX, see MCP Compatibility with MCP VERSIONS 2.5 THR

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [20]:
model.save_pretrained("gemma_4_lora")  # Local saving
tokenizer.save_pretrained("gemma_4_lora")
# model.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving

['gemma_4_lora/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [21]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma_4_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

ValueError: Cannot use apply_chat_template because this processor does not have a chat template.

### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-4-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-4-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-finetune", tokenizer,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )

Now, use the `gemma-4-finetune.gguf` file or `gemma-4-finetune-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).